In [1]:
import os
import sys
import json
import glob
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
sys.path.append("..")
from process.ifc_analysis_utils import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', None)

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Times New Roman"

In [2]:
root = Path.cwd().parent
mixs = ["mix_37_235", "mix_55_532"]
metrics = ["F1_score", "G_mean"]

all_results = build_all_results(data_root= root/"experiments")
artifacts_table = build_artifacts_path_table(data_root=root/"experiments")

orig_table = build_original_data_path_table(data_root=root/"experiments")
orig_table = add_pos_paths(orig_table, pos_label=1)

✅ build_all_results 完成: 共11625行, baseline=3255, none=930, self_loop=7440


In [3]:
from utils.common import DATASETS, CLASSIFIERS
all_datasets = DATASETS
num_datasets = ['breast_cancer_coimbra', 'PIMA_diabetes']
cat_datasets = ['TCGA_InfoWithGrade', 'Thyroid_Diff','diabetes_risk_prediction']
mixed_datasets = ['framingham', 'hepatitis']

my_datasets = all_datasets
my_classifiers = CLASSIFIERS #["SVC", "MLP", "XGB"]

In [ ]:
for mix in mixs:
    for metric in metrics:
        print(f"Results of {mix}, in metric {metric} in run_main_ablation:")
        res_A = run_main_ablation(all_results, metric=metric, mix_ratio=mix, datasets=my_datasets, classifiers=my_classifiers)
        print(res_A['cross_rank'])
        print(f"omnibus_stat: {res_A['omnibus_stat']}, omnibus_p: {res_A['omnibus_p']}")
        display(res_A['nemenyi'])

In [ ]:
# B. Stage-depth消融(2 vs 3)
for mix in mixs:
    for metric in metrics:
        print(f"Results of {mix}, in metric {metric}")
        view = select_mix_ratio_view(all_results, mix)
        res_B = paired_test(view, metric=metric, compare_col="stage_depth", 
                            level_a=2, level_b=3, datasets=my_datasets, classifiers=my_classifiers)
        display(res_B["summary"])
        print(f"overall_stat: {res_B['overall_stat']}, overall_p: {res_B['overall_p']}")

In [ ]:
for mix in mixs:
    for metric in metrics:
        view = select_mix_ratio_view(all_results, mix_ratio=mix)
        res_G = paired_test(view, metric=metric, compare_col="stage_depth", 
                            level_a=2, level_b=3, datasets=my_datasets, classifiers=my_classifiers)
        res_G["block_results"]["diff"].describe()
        print(mix, metric, ((res_G["block_results"]["diff"] > 0).sum(), (res_G["block_results"]["diff"] < 0).sum()))

In [ ]:
# C. Mix_ratio敏感性
for metric in metrics:
    for stage_depth in [2, 3]:
        print(f"mix_ratio 敏感性分析， {metric}指标， stage_depth: {stage_depth}")
        res_C = paired_test(all_results, metric=metric, compare_col="mix_ratio", 
                            level_a="mix_37_235", level_b="mix_55_532",
                            fixed_filters={"stage_depth": stage_depth}, 
                            datasets=my_datasets, classifiers=my_classifiers)
        display(res_C["summary"])

In [ ]:
# D. smote自循环深度趋势(1→2→3),锁定一个mix_ratio

def to_dataframe(res):
    rows = []
    for cfg, vals in res.items():
        rows.append({
            'config': cfg,
            'd1_mean': vals['means']['d1'],
            'd2_mean': vals['means']['d2'],
            'd3_mean': vals['means']['d3'], 
            'n_blocks': vals['n_blocks'],
            'p_increasing': vals['p_increasing'],
            'p_decreasing': vals['p_decreasing'],
        })
    df_res = pd.DataFrame(rows)
    return df_res

for mix in mixs:
    for metric in metrics:
        print(f"smote 自循环深度趋势分析，{mix}, {metric}")
        res_D = smote_depth_trend(all_results, metric=metric, mix_ratio=mix, 
                                  datasets=cat_datasets, classifiers=my_classifiers)
        display(to_dataframe(res_D))

## 分类结果展示

In [ ]:
raw_table = build_raw_reference_table(all_results)

vae_only_summary = summarize_method_stats(all_results, "none_noft")   # 你要的独立汇总

extra_rows = [
    ("VAE-only", "none_noft", None, None),  
    ("VAE+FT",     "none_ft",   None, None),
    ("IFC",       "smote_ft",  "mix_55_532", 3),
]
main_table = build_comparison_table(all_results, extra_rows, mark_close_calls=True)

## 4. TSNE 可视化

In [ ]:
import importlib
import process.visualization
importlib.reload(process.visualization)

In [ ]:
from utils.common import load_json
from process.visualization import *

dataset = "framingham"
seed, fold = pick_representative_fold(all_results, dataset)
orig_row = orig_table[(orig_table["Dataset"]==dataset) & (orig_table["Seed"]==seed) & (orig_table["Fold"]==fold)].iloc[0]

datasets_info = load_json(root / "data/datasets_info.json")

numerical_cols = datasets_info[dataset]["numerical"]
categorical_cols = datasets_info[dataset]["categorical"]

feature_cols = numerical_cols + categorical_cols
X_train_real = pd.read_parquet(orig_row["X_train_path"])
preprocessor = build_preprocessor(X_train_real, numerical_cols, categorical_cols)  # 有现成encoder的话优先换成那个
real_df = pd.read_parquet(orig_row["X_train_pos_path"])

In [ ]:
group_specs = [
    ("VAE-only", load_synthetic_stage(artifacts_table, dataset, seed, fold, "none_noft", stage="v0")),
    ("VAE + FT", load_synthetic_stage(artifacts_table, dataset, seed, fold, "none_ft", stage="__all_clusters__")),
    ("Self-Circulating v1", load_synthetic_stage(artifacts_table, dataset, seed, fold, "smote_ft", "mix_55_532", 3, "v1")),
    ("Self-Circulating v2", load_synthetic_stage(artifacts_table, dataset, seed, fold, "smote_ft", "mix_55_532", 3, "v2")),
    ("Self-Circulating v3", load_synthetic_stage(artifacts_table, dataset, seed, fold, "smote_ft", "mix_55_532", 3, "v3")),
    ("IFC (SMOTE_FT)", load_synthetic_stage(artifacts_table, dataset, seed, fold, "smote_ft", "mix_55_532", 3, "__fidelity__")),
] + [
    (m, load_synthetic_stage(artifacts_table, dataset, seed, fold, m, stage="__fidelity__"))
    for m in ["smote", "ctgan", "tvae", "forestdiffusion", "tabddpm", "tabkde"]
]

combined = build_combined_embedding(real_df, group_specs, feature_cols, preprocessor)


fig_evo = render_subplot_grid(
    combined, labels=["VAE-only", "VAE + FT", "Self-Circulating v1", 
                      "Self-Circulating v2", "Self-Circulating v3", "IFC (SMOTE_FT)"],
    show_labels=["VAE-only", "VAE + FT", "Self-Circulating v1", 
                 "Self-Circulating v2", "Self-Circulating v3", "IFC (SMOTE_FT)"],)

fig_base = render_subplot_grid(
    combined, labels=["smote", "ctgan", "tvae", "forestdiffusion", "tabddpm", "tabkde"], #, "IFC (final)"
    show_labels=["SMOTE", "CTGAN", "TVAE", "ForestDiffusion", "TabDDPM", "TabKDE"],)

fig_evo.savefig(f'{dataset}_ifc_evo_tsne.pdf', bbox_inches='tight')
fig_base.savefig(f'{dataset}_base_tsne.pdf', bbox_inches='tight')

## 5. 训练数据分布斜度和多峰检测等

In [ ]:
from scipy.spatial.distance import jensenshannon
from scipy.stats import skew, gaussian_kde
from scipy.signal import find_peaks
import numpy as np, pandas as pd


def numeric_peak_count(data, prominence_ratio=0.05, pad_ratio=0.15, grid_size=500):
    """从你原来的plot_kde_with_peaks里剥出纯数值版本,不画图,方便批量跑很多阶段。"""
    data = np.asarray(data).flatten()
    data = data[~np.isnan(data)]
    if len(data) < 5:
        return np.nan
    x_min, x_max = data.min(), data.max()
    x_range = x_max - x_min if x_max > x_min else 1.0
    x_grid = np.linspace(x_min - pad_ratio*x_range, x_max + pad_ratio*x_range, grid_size)
    y = gaussian_kde(data)(x_grid)
    peaks, _ = find_peaks(y, prominence=prominence_ratio * y.max())
    return len(peaks)


def categorical_shift(real_series, synthetic_series):
    """
    类别列的"skewness替代品",对应数值列的偏度:
    - JS散度(相对real,0=一致,1=完全不同)——衡量分布形状的偏移
    - 类别覆盖率(synthetic出现的类别数/real出现的类别数,<1说明发生了"类别坍缩",
      是manifold stiffening在类别特征上的对应体现)
    """
    real_counts = real_series.value_counts(normalize=True)
    syn_counts = synthetic_series.value_counts(normalize=True)
    all_cats = real_counts.index.union(syn_counts.index)
    p = real_counts.reindex(all_cats, fill_value=0).values
    q = syn_counts.reindex(all_cats, fill_value=0).values
    js = jensenshannon(p, q, base=2)
    coverage = syn_counts.index.nunique() / max(real_counts.index.nunique(), 1)
    return round(js, 3), round(coverage, 3)


def build_stage_evolution_table(real_df, stage_frames, numerical_cols, categorical_cols):
    """
    stage_frames: [(label, df), ...],比如[("v1", df_v1), ("v2", df_v2), ("v3", df_v3)]
    (real_df单独传,作为每个阶段的对照基准,不放进stage_frames里)。
    输出逐阶段、逐列的skewness/峰值个数(数值)、JS散度/类别覆盖率(类别)——
    这张表同时支撑manifold stiffening(峰值/覆盖率是否随阶段下降)和
    structural anchoring(不同mix_ratio下JS散度/skewness偏移轨迹是否不同)。
    """
    rows = []
    real_row_base = {"stage": "Real"}
    for col in numerical_cols:
        rows.append({**real_row_base, "column": col, "type": "numeric",
                     "skewness": round(skew(real_df[col].dropna()), 3),
                     "n_peaks": numeric_peak_count(real_df[col])})
    for label, df in stage_frames:
        for col in numerical_cols:
            rows.append({"stage": label, "column": col, "type": "numeric",
                         "skewness": round(skew(df[col].dropna()), 3),
                         "n_peaks": numeric_peak_count(df[col])})
        for col in categorical_cols:
            js, coverage = categorical_shift(real_df[col], df[col])
            rows.append({"stage": label, "column": col, "type": "categorical",
                         "js_divergence": js, "category_coverage": coverage})
    return pd.DataFrame(rows)


# 跑法(接上一轮已经load好的group_specs):
stage_frames = [(label, df) for label, df in group_specs if label.startswith("Self-loop") or label == "IFC (final)"]
evolution_table = build_stage_evolution_table(real_df, stage_frames, numerical_cols, categorical_cols)

In [ ]:
own_pipeline_labels = ["VAE only", "VAE + FT", "Self-loop v1", "Self-loop v2", "Self-loop v3", "IFC (final)"]
stage_frames = [(label, df) for label, df in group_specs if label in own_pipeline_labels]
evolution_table = build_stage_evolution_table(real_df, stage_frames, numerical_cols, categorical_cols)

In [ ]:
def compute_composite_fidelity(real_df, synthetic_df, numerical_cols, categorical_cols):
    real_skew = {c: skew(real_df[c].dropna()) for c in numerical_cols}
    real_peaks = {c: numeric_peak_count(real_df[c]) for c in numerical_cols}
    num_dev, peak_mismatch, cat_js = [], [], []
    for c in numerical_cols:
        num_dev.append(abs(skew(synthetic_df[c].dropna()) - real_skew[c]))
        peak_mismatch.append(int(numeric_peak_count(synthetic_df[c]) != real_peaks[c]))
    for c in categorical_cols:
        js, _ = categorical_shift(real_df[c], synthetic_df[c])
        cat_js.append(js)
    return {
        "mean_abs_skew_dev": np.mean(num_dev) if num_dev else np.nan,
        "peak_mismatch_rate": np.mean(peak_mismatch) if peak_mismatch else np.nan,
        "mean_js_divergence": np.mean(cat_js) if cat_js else np.nan,
    }


def build_fidelity_summary(all_results, orig_table, artifacts_table, datasets, methods_to_show,
                            datasets_info):
    """methods_to_show: {label: spec}(同load_synthetic_stage的spec),跑一遍所有数据集,
    产出Dataset×Method的复合fidelity表——baseline和IFC自身阶段用同一套spec机制混在一起传。"""
    rows = []
    for ds in datasets:
        numerical_cols = datasets_info[ds]["numerical"]
        categorical_cols = datasets_info[ds]["categorical"]

        seed, fold = pick_representative_fold(all_results, ds)
        orig_row = orig_table[(orig_table["Dataset"]==ds)&(orig_table["Seed"]==seed)&(orig_table["Fold"]==fold)].iloc[0]
        real_df = pd.read_parquet(orig_row["X_train_pos_path"])
        for label, spec in methods_to_show.items():
            syn_df = load_synthetic_stage(artifacts_table, ds, seed, fold, **spec)
            rows.append({"Dataset": ds, "Method": label,
                         **compute_composite_fidelity(real_df, syn_df, numerical_cols, categorical_cols)})
    return pd.DataFrame(rows)


def plot_fidelity_heatmap(summary_df, value_col, title=""):
    """行=Dataset,列=Method,一眼看出哪个数据集/哪个方法拟真度差——数值越小(除peak_mismatch_rate)越好。"""
    pivot = summary_df.pivot(index="Dataset", columns="Method", values=value_col)
    fig, ax = plt.subplots(figsize=(1.2*len(pivot.columns)+2, 0.6*len(pivot.index)+2))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    ax.set_title(title)
    fig.tight_layout()
    return fig


def summarize_across_datasets(summary_df, value_col):
    """跨数据集汇总成一张Method排名表,用来在讨论里直接对比IFC和6个baseline。"""
    return summary_df.groupby("Method")[value_col].agg(["mean", "std"]).sort_values("mean")

In [ ]:
methods_to_show = {
    "IFC (final)": {"method": "smote_ft", "mix_ratio": "mix_55_532", "stage_depth": 3, "stage": "__fidelity__"},
    "SMOTE": {"method": "smote", "stage": "__fidelity__"},
    "CTGAN": {"method": "ctgan", "stage": "__fidelity__"},
    "TVAE": {"method": "tvae", "stage": "__fidelity__"},
    "ForestDiff": {"method": "forestdiffusion", "stage": "__fidelity__"},
    "TabDDPM": {"method": "tabddpm", "stage": "__fidelity__"},
    "TabKDE": {"method": "tabkde", "stage": "__fidelity__"},
}
fidelity_summary = build_fidelity_summary(all_results, orig_table, artifacts_table, DATASETS, methods_to_show,
                                           datasets_info)
plot_fidelity_heatmap(fidelity_summary, "mean_abs_skew_dev", title="Skewness deviation from real")
summarize_across_datasets(fidelity_summary, "mean_js_divergence")

### 5.1 训练数据分布统计

In [ ]:
from scipy.spatial.distance import jensenshannon
from scipy.stats import skew
import numpy as np, pandas as pd

def categorical_imbalance(series):
    """类别列的'偏度类比',只描述real data本身,不需要synthetic:
    JS散度(相对均匀分布),0=完全均匀,越大越不均衡。"""
    counts = series.value_counts(normalize=True)
    k = len(counts)
    if k <= 1:
        return 0.0
    uniform = np.full(k, 1.0 / k)
    return round(jensenshannon(counts.values, uniform, base=2), 3)


def real_data_distribution_summary(datasets_dict, numerical_cols_map, categorical_cols_map):
    """
    datasets_dict: {dataset_name: real_df}(建议用X_train_pos,毕竟你算法目标就是少数类,
    "少数类本身分布复杂"是比"全量数据分布复杂"更贴题的动机)
    numerical_cols_map/categorical_cols_map: {dataset_name: [cols]} —— 纯数值/纯类别数据集
    某一类为空时,对应列留NaN,不强行填0或跳过整行。
    """
    rows = []
    for ds, df in datasets_dict.items():
        num_cols, cat_cols = numerical_cols_map.get(ds, []), categorical_cols_map.get(ds, [])
        row = {"Dataset": ds, "n_numeric": len(num_cols), "n_categorical": len(cat_cols)}
        if num_cols:
            skews = [abs(skew(df[c].dropna())) for c in num_cols]
            row["mean_abs_skew"] = round(np.mean(skews), 3)
            row["pct_severe_skew"] = round(np.mean([s > 2 for s in skews]), 3)
            row["pct_multimodal"] = round(np.mean([numeric_peak_count(df[c]) >= 2 for c in num_cols]), 3)
        else:
            row["mean_abs_skew"] = row["pct_severe_skew"] = row["pct_multimodal"] = np.nan
        row["mean_cat_imbalance"] = round(np.mean([categorical_imbalance(df[c]) for c in cat_cols]), 3) if cat_cols else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# ---- 第一步: 标记real data里"难建模"的列(复用你已有的numeric_peak_count/categorical_imbalance) ----
def flag_real_problem_columns(real_df, numerical_cols, categorical_cols,
                               skew_threshold=1, imbalance_threshold=0.3):
    """
    逐列标记:不是"要消除的问题",而是后面检验synthetic有没有更好保留这些特征的依据。
    数值列: |skewness|>阈值 或 多峰 → flagged=True
    类别列: 相对均匀分布的JS散度>阈值(高度不均衡) → flagged=True
    """
    rows = []
    for c in numerical_cols:
        s = skew(real_df[c].dropna())
        n_peaks = numeric_peak_count(real_df[c])
        rows.append({"column": c, "type": "numeric", "skewness": round(s, 3),
                     "n_peaks": n_peaks, "flagged": abs(s) > skew_threshold or n_peaks >= 2})
    for c in categorical_cols:
        js = categorical_imbalance(real_df[c])
        rows.append({"column": c, "type": "categorical", "js_vs_uniform": js,
                     "flagged": js > imbalance_threshold})
    return pd.DataFrame(rows)


# ---- 第二步: 只在这些"困难列"上,比较各个synthetic方法对real的保留程度(复用categorical_shift) ----
def build_mitigation_table(real_df, problem_cols_df, group_specs):
    """
    group_specs: [(label, df), ...] —— IFC final + 6个baseline,和之前t-SNE/相关性分析用的是同一份
    只统计flagged==True的列: 数值列比skewness偏差+峰值个数是否对上,类别列比JS散度。
    数值越小(peak_match_rate越大)=保留得越好。
    """
    flagged = problem_cols_df[problem_cols_df["flagged"]]
    rows = []
    for label, syn_df in group_specs:
        num_devs, peak_matches, cat_js = [], [], []
        for _, r in flagged.iterrows():
            c = r["column"]
            if r["type"] == "numeric":
                num_devs.append(abs(skew(syn_df[c].dropna()) - r["skewness"]))
                peak_matches.append(int(numeric_peak_count(syn_df[c]) == r["n_peaks"]))
            else:
                js, _ = categorical_shift(real_df[c], syn_df[c])
                cat_js.append(js)
        rows.append({
            "group": label,
            "mean_skew_dev_on_flagged": round(np.mean(num_devs), 3) if num_devs else np.nan,
            "peak_match_rate_on_flagged": round(np.mean(peak_matches), 3) if peak_matches else np.nan,
            "mean_js_on_flagged_categorical": round(np.mean(cat_js), 3) if cat_js else np.nan,
        })
    return pd.DataFrame(rows)


# ---- 第三步: 跨数据集跑一遍,拼成一张总表(复用pick_representative_fold/load_synthetic_stage) ----
def build_mitigation_summary_all_datasets(all_results, orig_table, artifacts_table, datasets,
                                           methods_to_show, numerical_cols_map, categorical_cols_map):
    """
    methods_to_show: 和你上一轮fidelity_summary用的是同一个dict(IFC final + 6 baseline的spec)。
    numerical_cols_map/categorical_cols_map: {dataset_name: [列名...]},每个数据集的数值/类别列不同,
    某个数据集某一类为空(纯数值或纯类别数据集)时,对应指标自然是NaN,不强行填0。
    """
    all_rows = []
    for ds in datasets:
        seed, fold = pick_representative_fold(all_results, ds)
        orig_row = orig_table[(orig_table["Dataset"] == ds) & (orig_table["Seed"] == seed)
                               & (orig_table["Fold"] == fold)].iloc[0]
        real_df = pd.read_parquet(orig_row["X_train_pos_path"])
        num_cols = numerical_cols_map.get(ds, [])
        cat_cols = categorical_cols_map.get(ds, [])

        problem_cols = flag_real_problem_columns(real_df, num_cols, cat_cols)
        group_specs_ds = [(label, load_synthetic_stage(artifacts_table, ds, seed, fold, **spec))
                           for label, spec in methods_to_show.items()]
        mitigation = build_mitigation_table(real_df, problem_cols, group_specs_ds)
        mitigation.insert(0, "Dataset", ds)
        all_rows.append(mitigation)
    return pd.concat(all_rows, ignore_index=True).rename(columns={"group": "Method"})


# ---- 跑法 ----
methods_to_show = {   # 和你上一轮fidelity_summary里的定义完全一样,直接复用
    "IFC (final)": {"method": "smote_ft", "mix_ratio": "mix_55_532", "stage_depth": 2, "stage": "__fidelity__"},
    "SMOTE": {"method": "smote", "stage": "__fidelity__"},
    "CTGAN": {"method": "ctgan", "stage": "__fidelity__"},
    "TVAE": {"method": "tvae", "stage": "__fidelity__"},
    "ForestDiff": {"method": "forestdiffusion", "stage": "__fidelity__"},
    "TabDDPM": {"method": "tabddpm", "stage": "__fidelity__"},
    "TabKDE": {"method": "tabkde", "stage": "__fidelity__"},
}
numerical_cols_map = {name: info['numerical'] for name, info in datasets_info.items()} 
categorical_cols_map = {name: info['categorical'] for name, info in datasets_info.items()} 
mitigation_all = build_mitigation_summary_all_datasets(
    all_results, orig_table, artifacts_table, DATASETS, methods_to_show,
    numerical_cols_map, categorical_cols_map,
)

# 复用上一轮的热力图/排名函数,只是换了value_col
plot_fidelity_heatmap(mitigation_all, "mean_skew_dev_on_flagged", title="mean_skew_dev_on_flagged")
plot_fidelity_heatmap(mitigation_all, "mean_js_on_flagged_categorical", title="mean_js_on_flagged_categorical")
summarize_across_datasets(mitigation_all, "mean_skew_dev_on_flagged")   # IFC vs 6个baseline的跨数据集排名

## 6. Kendall可视化

In [ ]:
from utils.common import load_json
from process.visualization import *

dataset = "framingham"
seed, fold = pick_representative_fold(all_results, dataset)
orig_row = orig_table[(orig_table["Dataset"]==dataset) & (orig_table["Seed"]==seed) & (orig_table["Fold"]==fold)].iloc[0]

datasets_info = load_json(root / "data/datasets_info.json")

numerical_cols = datasets_info[dataset]["numerical"]
categorical_cols = datasets_info[dataset]["categorical"]

feature_cols = numerical_cols + categorical_cols
X_train_real = pd.read_parquet(orig_row["X_train_path"])
preprocessor = build_preprocessor(X_train_real, numerical_cols, categorical_cols)  # 有现成encoder的话优先换成那个
real_df = pd.read_parquet(orig_row["X_train_pos_path"])

In [ ]:
from scipy.stats import kendalltau
import matplotlib.pyplot as plt
import seaborn as sns

def kendall_corr_matrix(df, numerical_cols):
    n = len(numerical_cols)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            tau, _ = kendalltau(df[numerical_cols[i]], df[numerical_cols[j]])
            mat[i, j] = mat[j, i] = tau
    return pd.DataFrame(mat, index=numerical_cols, columns=numerical_cols)


def plot_correlation_grid(real_df, group_specs, numerical_cols, title=""):
    """real单独一张 + group_specs(IFC各阶段+6个baseline统一放在一起,和t-SNE一样保持可比)各一张,
    颜色刻度全部统一(-1,1)。"""
    n_total = len(group_specs) + 1
    ncols = int(np.ceil(np.sqrt(n_total)))
    nrows = int(np.ceil(n_total / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 3*nrows))
    axes = np.array(axes).reshape(-1)

    real_corr = kendall_corr_matrix(real_df, numerical_cols)
    sns.heatmap(real_corr, ax=axes[0], cmap="RdBu_r", vmin=-1, vmax=1, cbar=False, square=True,
                xticklabels=False, yticklabels=True)
    axes[0].set_title("Real", fontsize=10)

    for ax, (label, df) in zip(axes[1:], group_specs):
        sns.heatmap(kendall_corr_matrix(df, numerical_cols), ax=ax, cmap="RdBu_r", vmin=-1, vmax=1,
                    cbar=False, square=True, xticklabels=False, yticklabels=False)
        ax.set_title(label, fontsize=10)
    for ax in axes[n_total:]:
        ax.axis("off")
    fig.suptitle(title); fig.tight_layout()
    return fig


def plot_correlation_deviation_grid(real_df, group_specs, numerical_cols, title=""):
    """更直接:每组画(synthetic_corr - real_corr),子图标题里带上均值绝对偏差(MAD)。"""
    real_corr = kendall_corr_matrix(real_df, numerical_cols)
    n = len(group_specs)
    #ncols = int(np.ceil(np.sqrt(n))); nrows = int(np.ceil(n / ncols))
    ncols = 6; nrows = 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows))
    axes = np.array(axes).reshape(-1)
    iu = np.triu_indices(len(numerical_cols), k=1)

    for ax, (label, df) in zip(axes, group_specs):
        diff = kendall_corr_matrix(df, numerical_cols) - real_corr
        sns.heatmap(diff, ax=ax, cmap="RdBu_r", vmin=-1, vmax=1, cbar=False, square=False,
                    xticklabels=False, yticklabels=False)
        mad = diff.values[iu].__abs__().mean()
        ax.set_title(f"{label}\n(MAD={mad:.3f})", fontsize=12)
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title + " — deviation from Real"); fig.tight_layout()
    fig.tight_layout()
    return fig


def correlation_deviation_summary(real_df, group_specs, numerical_cols):
    """标量汇总(上三角绝对差均值),和你fidelity_summary一样的跨数据集/跨方法表结构,放Discussion里对比IFC vs 6个baseline。"""
    real_corr = kendall_corr_matrix(real_df, numerical_cols)
    iu = np.triu_indices(len(numerical_cols), k=1)
    rows = []
    for label, df in group_specs:
        diff = np.abs((kendall_corr_matrix(df, numerical_cols) - real_corr).values[iu])
        rows.append({"group": label, "mean_abs_corr_dev": round(diff.mean(), 3), "max_abs_corr_dev": round(diff.max(), 3)})
    return pd.DataFrame(rows)


fig1 = plot_correlation_grid(real_df, group_specs, numerical_cols, title=f"{dataset}: Kendall correlation")
fig2 = plot_correlation_deviation_grid(real_df, group_specs, numerical_cols, title=f"{dataset}")
corr_summary = correlation_deviation_summary(real_df, group_specs, numerical_cols)